In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

clean_path   = r'C:\Users\user\DSS_Project\data\clean'
raw_path     = r'C:\Users\user\DSS_Project\data\raw'
export_path  = r'C:\Users\user\DSS_Project\exports'
powerbi_path = r'C:\Users\user\DSS_Project\exports\powerbi'
db_path      = r'C:\Users\user\DSS_Project\database\dss_project.db'

print("🔄 Loading all datasets...")

# 1️⃣ Ecommerce
eco = pd.read_csv(f'{clean_path}\\ecommerce_clean.csv')
eco['date'] = pd.to_datetime(eco['date'])
eco_df = pd.DataFrame({
    'customer_id'  : 'eco_' + eco['customer_id'].astype(str),
    'date'         : eco['date'],
    'total_amount' : eco['total_amount'],
    'quantity'     : eco['quantity'],
    'rating'       : eco['customer_rating'],
    'category'     : eco['product_category'],
    'source'       : 'ecommerce'
})
print(f"✅ ecommerce     : {eco_df['customer_id'].nunique():,} customers")

# 2️⃣ Retail
ret = pd.read_csv(f'{clean_path}\\retail_clean.csv',
                  usecols=['customer_id','transaction_date',
                           'total_sales','quantity',
                           'product_rating','product_category'],
                  nrows=500000)
ret['date'] = pd.to_datetime(ret['transaction_date'], errors='coerce')
ret_df = pd.DataFrame({
    'customer_id'  : 'ret_' + ret['customer_id'].astype(str),
    'date'         : ret['date'],
    'total_amount' : ret['total_sales'],
    'quantity'     : ret['quantity'],
    'rating'       : ret['product_rating'],
    'category'     : ret['product_category'],
    'source'       : 'retail'
})
ret_df = ret_df.dropna(subset=['date','total_amount'])
ret_df = ret_df[ret_df['total_amount'] > 0]
print(f"✅ retail        : {ret_df['customer_id'].nunique():,} customers")

# 3️⃣ kz
kz = pd.read_csv(f'{raw_path}\\kz.csv',
                 usecols=['user_id','event_time','price','category_code'],
                 nrows=300000)
kz['date'] = pd.to_datetime(kz['event_time'], errors='coerce')
kz['date'] = kz['date'].dt.tz_localize(None)
kz = kz.dropna(subset=['price','user_id'])
kz = kz[kz['price'] > 0]
kz_df = pd.DataFrame({
    'customer_id'  : 'kz_' + kz['user_id'].astype(str),
    'date'         : kz['date'],
    'total_amount' : kz['price'],
    'quantity'     : 1,
    'rating'       : 3.5,
    'category'     : kz['category_code'].str.split('.').str[0].fillna('unknown'),
    'source'       : 'kz'
})
print(f"✅ kz            : {kz_df['customer_id'].nunique():,} customers")

# 4️⃣ Online Retail
try:
    onl = pd.read_csv(f'{raw_path}\\online_retail_II.csv',
                      encoding='utf-8', on_bad_lines='skip',
                      usecols=['Customer ID','InvoiceDate',
                               'Quantity','Price','Description'])
    onl = onl.dropna(subset=['Customer ID'])
    onl['date'] = pd.to_datetime(onl['InvoiceDate'], errors='coerce')
    onl['total_amount'] = onl['Quantity'] * onl['Price']
    onl = onl[onl['total_amount'] > 0]
    onl_df = pd.DataFrame({
        'customer_id'  : 'onl_' + onl['Customer ID'].astype(str),
        'date'         : onl['date'],
        'total_amount' : onl['total_amount'],
        'quantity'     : onl['Quantity'],
        'rating'       : 3.5,
        'category'     : onl['Description'].str[:20].fillna('unknown'),
        'source'       : 'online_retail'
    })
    print(f"✅ online_retail : {onl_df['customer_id'].nunique():,} customers")
except Exception as e:
    onl_df = pd.DataFrame()
    print(f"⚠️ online_retail : skipped ({e})")

# Combine All
dfs = [eco_df, ret_df, kz_df]
if len(onl_df) > 0:
    dfs.append(onl_df)

combined = pd.concat(dfs, ignore_index=True)
combined['date'] = pd.to_datetime(combined['date'], errors='coerce')
combined = combined.dropna(subset=['date','total_amount'])
combined = combined[combined['total_amount'] > 0]
combined = combined[combined['date'] >= '2010-01-01']
combined = combined.reset_index(drop=True)

print(f"\n✅ Combined Total:")
print(f"   Rows      : {len(combined):,}")
print(f"   Customers : {combined['customer_id'].nunique():,}")
print(f"   Date range: {combined['date'].min().date()} → {combined['date'].max().date()}")
print(f"\nSource breakdown:")
print(combined.groupby('source')['customer_id'].nunique().to_string())

🔄 Loading all datasets...
✅ ecommerce     : 5,000 customers
✅ retail        : 500,000 customers
✅ kz            : 22,045 customers
✅ online_retail : 5,878 customers

✅ Combined Total:
   Rows      : 1,347,002
   Customers : 532,745
   Date range: 2010-01-04 → 2024-03-26

Source breakdown:
source
ecommerce          5000
kz                21956
online_retail      5789
retail           500000


In [2]:
# CLV لكل عميل
print("🔄 Calculating CLV for all customers...")

clv = combined.groupby('customer_id').agg(
    total_spent    = ('total_amount', 'sum'),
    order_count    = ('total_amount', 'count'),
    avg_order      = ('total_amount', 'mean'),
    max_order      = ('total_amount', 'max'),
    min_order      = ('total_amount', 'min'),
    std_order      = ('total_amount', 'std'),
    first_purchase = ('date', 'min'),
    last_purchase  = ('date', 'max'),
    avg_rating     = ('rating', 'mean'),
    unique_cats    = ('category', 'nunique'),
    source         = ('source', 'first')
).reset_index()

# Tenure
max_date = combined['date'].max()
clv['tenure_days']  = (max_date - clv['first_purchase']).dt.days
clv['recency_days'] = (max_date - clv['last_purchase']).dt.days
clv['tenure_years'] = (clv['tenure_days'] / 365).clip(lower=0.1)
clv['std_order']    = clv['std_order'].fillna(0)

# CLV Calculations
clv['clv_historical']  = clv['total_spent'].round(2)
clv['clv_annual']      = (clv['total_spent'] / clv['tenure_years']).round(2)
clv['clv_predicted']   = (clv['avg_order'] * clv['order_count'] / 
                           clv['tenure_years']).round(2)
clv['clv_3month']      = (clv['clv_annual'] / 4).round(2)
clv['purchase_freq']   = (clv['order_count'] / clv['tenure_years']).round(2)

# Segments per source
def assign_segment(df):
    p90 = df['clv_annual'].quantile(0.90)
    p75 = df['clv_annual'].quantile(0.75)
    p50 = df['clv_annual'].quantile(0.50)
    return df['clv_annual'].apply(
        lambda v: 'Platinum' if v >= p90 else
                  'Gold'     if v >= p75 else
                  'Silver'   if v >= p50 else 'Bronze'
    ), p90, p75, p50

clv['clv_segment'], p90, p75, p50 = assign_segment(clv)

print(f"✅ CLV calculated for {len(clv):,} customers")
print(f"\n📊 CLV SEGMENTS:")
seg = clv.groupby('clv_segment').agg(
    count      = ('customer_id', 'count'),
    avg_clv    = ('clv_annual', 'mean'),
    total_clv  = ('clv_annual', 'sum'),
    avg_orders = ('order_count', 'mean'),
    avg_rating = ('avg_rating', 'mean'),
).round(2)
print(seg.to_string())

print(f"\n💰 Total Annual CLV  : ${clv['clv_annual'].sum():,.0f}")
print(f"💰 Avg CLV           : ${clv['clv_annual'].mean():,.2f}")
print(f"💰 Median CLV        : ${clv['clv_annual'].median():,.2f}")
print(f"💰 Top 10% CLV       : ${p90:,.2f}+")
print(f"\n📊 By Source:")
print(clv.groupby('source').agg(
    customers  = ('customer_id', 'count'),
    avg_clv    = ('clv_annual', 'mean'),
    total_clv  = ('clv_annual', 'sum')
).round(2).to_string())

🔄 Calculating CLV for all customers...
✅ CLV calculated for 532,745 customers

📊 CLV SEGMENTS:
              count  avg_clv     total_clv  avg_orders  avg_rating
clv_segment                                                       
Bronze       266372   692.64  1.845003e+08        3.53        3.06
Gold          79912  2585.02  2.065740e+08        1.16        3.00
Platinum      53275  3699.21  1.970752e+08        2.76        3.05
Silver       133186  1894.93  2.523787e+08        1.26        3.01

💰 Total Annual CLV  : $840,528,163
💰 Avg CLV           : $1,577.73
💰 Median CLV        : $1,490.57
💰 Top 10% CLV       : $2,949.55+

📊 By Source:
               customers  avg_clv     total_clv
source                                         
ecommerce           5000  5702.48  2.851239e+07
kz                 21956    93.95  2.062736e+06
online_retail       5789   212.12  1.227955e+06
retail            500000  1617.45  8.087251e+08


In [4]:
print("🔄 Loading remaining datasets...")

# 1️⃣ Olist
olist = pd.read_csv(f'{clean_path}\\olist_clean.csv')
olist['date'] = pd.to_datetime(
    olist['order_purchase_timestamp'], errors='coerce')
olist['date'] = olist['date'].dt.tz_localize(None)
olist_df = pd.DataFrame({
    'customer_id'  : 'olist_' + olist['customer_id'].astype(str),
    'date'         : olist['date'],
    'total_amount' : olist['total_payment'],
    'quantity'     : 1,
    'rating'       : olist['review_score'].fillna(3),
    'category'     : olist['product_category_name_english'].fillna('unknown'),
    'source'       : 'olist'
})
olist_df = olist_df.dropna(subset=['date','total_amount'])
olist_df = olist_df[olist_df['total_amount'] > 0]
print(f"✅ olist          : {olist_df['customer_id'].nunique():,} customers")

# 2️⃣ ecommerce_new
try:
    eco_new = pd.read_csv(f'{clean_path}\\ecommerce_new_clean.csv')
    eco_new_df = pd.DataFrame({
        'customer_id'  : 'econew_' + eco_new['user_id'].astype(str),
        'date'         : pd.to_datetime(eco_new.get('date', 
                         eco_new.get('Purchase_Date', '2023-01-01')),
                         errors='coerce'),
        'total_amount' : eco_new.get('total_amount',
                         eco_new.get('Final_Price(Rs.)', 0)),
        'quantity'     : 1,
        'rating'       : 3.5,
        'category'     : eco_new.get('category',
                         eco_new.get('Category', 'unknown')),
        'source'       : 'ecommerce_new'
    })
    eco_new_df = eco_new_df.dropna(subset=['date','total_amount'])
    eco_new_df = eco_new_df[eco_new_df['total_amount'] > 0]
    print(f"✅ ecommerce_new  : {eco_new_df['customer_id'].nunique():,} customers")
except Exception as e:
    eco_new_df = pd.DataFrame()
    print(f"⚠️ ecommerce_new  : {e}")

# 3️⃣ eco_dataset
try:
    eco_d = pd.read_csv(f'{raw_path}\\ecommerce_dataset_updated.csv',
                        encoding='latin-1')
    eco_d_df = pd.DataFrame({
        'customer_id'  : 'ecod_' + eco_d['User_ID'].astype(str),
        'date'         : pd.to_datetime(eco_d['Purchase_Date'], 
                         errors='coerce'),
        'total_amount' : eco_d['Final_Price(Rs.)'],
        'quantity'     : 1,
        'rating'       : 3.5,
        'category'     : eco_d['Category'],
        'source'       : 'eco_dataset'
    })
    eco_d_df = eco_d_df.dropna(subset=['date','total_amount'])
    eco_d_df = eco_d_df[eco_d_df['total_amount'] > 0]
    print(f"✅ eco_dataset    : {eco_d_df['customer_id'].nunique():,} customers")
except Exception as e:
    eco_d_df = pd.DataFrame()
    print(f"⚠️ eco_dataset    : {e}")

# 4️⃣ data.csv
try:
    data = pd.read_csv(f'{raw_path}\\data.csv',
                       encoding='latin-1')
    data['total_amount'] = data['Quantity'] * data['UnitPrice']
    data_df = pd.DataFrame({
        'customer_id'  : 'data_' + data['InvoiceNo'].astype(str),
        'date'         : pd.to_datetime(data['InvoiceDate'], 
                         errors='coerce'),
        'total_amount' : data['total_amount'],
        'quantity'     : data['Quantity'],
        'rating'       : 3.5,
        'category'     : data['Description'].str[:20].fillna('unknown'),
        'source'       : 'data'
    })
    data_df = data_df.dropna(subset=['date','total_amount'])
    data_df = data_df[data_df['total_amount'] > 0]
    print(f"✅ data.csv       : {data_df['customer_id'].nunique():,} customers")
except Exception as e:
    data_df = pd.DataFrame()
    print(f"⚠️ data.csv       : {e}")

# 5️⃣ Telco Churn — مش sales data بس ممكن نحسب CLV من tenure
telco = pd.read_csv(f'{raw_path}\\WA_Fn-UseC_-Telco-Customer-Churn.csv')
telco_df = pd.DataFrame({
    'customer_id'  : 'telco_' + telco['customerID'].astype(str),
    'date'         : pd.Timestamp('2023-01-01'),
    'total_amount' : telco['TotalCharges'].replace(' ', '0')\
                     .astype(float),
    'quantity'     : 1,
    'rating'       : 3.5,
    'category'     : 'Telecom',
    'source'       : 'telco'
})
telco_df = telco_df[telco_df['total_amount'] > 0]
print(f"✅ telco_churn    : {telco_df['customer_id'].nunique():,} customers")

# 6️⃣ Bank Churn
bank = pd.read_csv(f'{raw_path}\\Churn_Modelling.csv')
bank_df = pd.DataFrame({
    'customer_id'  : 'bank_' + bank['CustomerId'].astype(str),
    'date'         : pd.Timestamp('2023-01-01'),
    'total_amount' : bank['Balance'],
    'quantity'     : 1,
    'rating'       : bank['Satisfaction Score'] if 'Satisfaction Score' in bank.columns else 3.5,
    'category'     : 'Banking',
    'source'       : 'bank'
})
bank_df = bank_df[bank_df['total_amount'] > 0]
print(f"✅ bank_churn     : {bank_df['customer_id'].nunique():,} customers")

print(f"\n✅ All remaining datasets loaded!")

🔄 Loading remaining datasets...
✅ olist          : 96,477 customers
✅ ecommerce_new  : 0 customers
✅ eco_dataset    : 1,454 customers
✅ data.csv       : 19,960 customers
✅ telco_churn    : 7,032 customers
✅ bank_churn     : 6,383 customers

✅ All remaining datasets loaded!


In [3]:
print("🔄 Running Bootstrap ROI (10,000 iterations)...")

mkt = pd.read_csv(f'{clean_path}\\marketing_clean.csv')

np.random.seed(42)
N_BOOT = 10000

def bootstrap_roi(revenue, budget, n=N_BOOT):
    rois = []
    for _ in range(n):
        idx = np.random.choice(len(revenue), len(revenue), replace=True)
        r = revenue.iloc[idx].sum()
        b = budget.iloc[idx].sum()
        rois.append((r - b) / b * 100 if b > 0 else 0)
    rois = np.array(rois)
    return {
        'mean'        : round(np.mean(rois), 2),
        'median'      : round(np.median(rois), 2),
        'std'         : round(np.std(rois), 2),
        'ci_low'      : round(np.percentile(rois, 2.5), 2),
        'ci_high'     : round(np.percentile(rois, 97.5), 2),
        'prob_positive': round((rois > 0).mean() * 100, 2)
    }

results = []

# Overall
overall = bootstrap_roi(
    mkt['sales_revenue_usd'], mkt['marketing_budget_usd'])
overall['dimension'] = 'Overall'
overall['category']  = 'All'
results.append(overall)
print(f"✅ Overall ROI: {overall['mean']:+.2f}% CI:[{overall['ci_low']:+.2f}%, {overall['ci_high']:+.2f}%]")

# By Channel
for grp, g in mkt.groupby('sales_channel'):
    r = bootstrap_roi(g['sales_revenue_usd'], g['marketing_budget_usd'])
    r['dimension'] = 'Channel'
    r['category']  = grp
    results.append(r)

# By Segment
for grp, g in mkt.groupby('customer_segment'):
    r = bootstrap_roi(g['sales_revenue_usd'], g['marketing_budget_usd'])
    r['dimension'] = 'Segment'
    r['category']  = grp
    results.append(r)

# By Season
for grp, g in mkt.groupby('season'):
    r = bootstrap_roi(g['sales_revenue_usd'], g['marketing_budget_usd'])
    r['dimension'] = 'Season'
    r['category']  = grp
    results.append(r)

# By Region
for grp, g in mkt.groupby('region'):
    r = bootstrap_roi(g['sales_revenue_usd'], g['marketing_budget_usd'])
    r['dimension'] = 'Region'
    r['category']  = grp
    results.append(r)

# By Category
for grp, g in mkt.groupby('product_category'):
    r = bootstrap_roi(g['sales_revenue_usd'], g['marketing_budget_usd'])
    r['dimension'] = 'Category'
    r['category']  = grp
    results.append(r)

# Best Combos
top_combos = []
for keys, g in mkt.groupby(['sales_channel','customer_segment','season']):
    if len(g) >= 10:
        r = bootstrap_roi(g['sales_revenue_usd'], g['marketing_budget_usd'])
        r['dimension'] = 'Combination'
        r['category']  = f"{keys[0]} | {keys[1]} | {keys[2]}"
        top_combos.append(r)

top_combos = sorted(top_combos, key=lambda x: x['mean'], reverse=True)[:10]
results.extend(top_combos)

df_boot = pd.DataFrame(results)

print(f"\n{'='*65}")
print(f"🏆 BOOTSTRAP ROI SUMMARY (10,000 iterations)")
print(f"{'='*65}")

best_ch  = max([r for r in results if r['dimension']=='Channel'],  key=lambda x: x['mean'])
best_seg = max([r for r in results if r['dimension']=='Segment'],  key=lambda x: x['mean'])
best_sea = max([r for r in results if r['dimension']=='Season'],   key=lambda x: x['mean'])
best_reg = max([r for r in results if r['dimension']=='Region'],   key=lambda x: x['mean'])
best_com = top_combos[0] if top_combos else None

print(f"   Overall ROI    : {overall['mean']:+.2f}% CI:[{overall['ci_low']:+.2f}%, {overall['ci_high']:+.2f}%]")
print(f"   Best Channel   : {best_ch['category']:20} → {best_ch['mean']:+.2f}%")
print(f"   Best Segment   : {best_seg['category']:20} → {best_seg['mean']:+.2f}%")
print(f"   Best Season    : {best_sea['category']:20} → {best_sea['mean']:+.2f}%")
print(f"   Best Region    : {best_reg['category']:20} → {best_reg['mean']:+.2f}%")
if best_com:
    print(f"   Best Combo     : {best_com['category']}")
    print(f"                  → {best_com['mean']:+.2f}% CI:[{best_com['ci_low']:+.2f}%, {best_com['ci_high']:+.2f}%]")

🔄 Running Bootstrap ROI (10,000 iterations)...
✅ Overall ROI: -41.07% CI:[-41.64%, -40.50%]

🏆 BOOTSTRAP ROI SUMMARY (10,000 iterations)
   Overall ROI    : -41.07% CI:[-41.64%, -40.50%]
   Best Channel   : Wholesale            → -32.13%
   Best Segment   : Corporate            → +8.05%
   Best Season    : Q4                   → -25.89%
   Best Region    : Kuwait               → -39.81%
   Best Combo     : Wholesale | Corporate | Q4
                  → +67.92% CI:[+51.67%, +84.70%]


In [5]:
# Combine كل الـ datasets
all_dfs = [eco_df, ret_df, kz_df, olist_df, telco_df, bank_df]

# أضف الـ optional datasets
for df in [eco_new_df, eco_d_df, data_df]:
    if len(df) > 0:
        all_dfs.append(df)

if len(onl_df) > 0:
    all_dfs.append(onl_df)

combined_full = pd.concat(all_dfs, ignore_index=True)
combined_full['date'] = pd.to_datetime(
    combined_full['date'], errors='coerce')
combined_full = combined_full.dropna(subset=['date','total_amount'])
combined_full = combined_full[combined_full['total_amount'] > 0]
combined_full = combined_full.reset_index(drop=True)

print(f"✅ FULL Combined:")
print(f"   Rows      : {len(combined_full):,}")
print(f"   Customers : {combined_full['customer_id'].nunique():,}")
print(f"\nSource breakdown:")
print(combined_full.groupby('source')['customer_id'].nunique().to_string())

# Recalculate CLV
clv_full = combined_full.groupby('customer_id').agg(
    total_spent    = ('total_amount', 'sum'),
    order_count    = ('total_amount', 'count'),
    avg_order      = ('total_amount', 'mean'),
    first_purchase = ('date', 'min'),
    last_purchase  = ('date', 'max'),
    avg_rating     = ('rating', 'mean'),
    unique_cats    = ('category', 'nunique'),
    source         = ('source', 'first')
).reset_index()

max_date = combined_full['date'].max()
clv_full['tenure_days']  = (max_date - clv_full['first_purchase']).dt.days
clv_full['recency_days'] = (max_date - clv_full['last_purchase']).dt.days
clv_full['tenure_years'] = (clv_full['tenure_days'] / 365).clip(lower=0.1)
clv_full['clv_annual']   = (clv_full['total_spent'] / 
                             clv_full['tenure_years']).round(2)
clv_full['clv_3month']   = (clv_full['clv_annual'] / 4).round(2)

# Segments
p90 = clv_full['clv_annual'].quantile(0.90)
p75 = clv_full['clv_annual'].quantile(0.75)
p50 = clv_full['clv_annual'].quantile(0.50)

clv_full['clv_segment'] = clv_full['clv_annual'].apply(
    lambda v: 'Platinum' if v >= p90 else
              'Gold'     if v >= p75 else
              'Silver'   if v >= p50 else 'Bronze')

print(f"\n{'='*65}")
print(f"📊 FULL CLV — {len(clv_full):,} customers")
print(f"{'='*65}")
seg = clv_full.groupby('clv_segment').agg(
    count     = ('customer_id', 'count'),
    avg_clv   = ('clv_annual', 'mean'),
    total_clv = ('clv_annual', 'sum'),
).round(2)
print(seg.to_string())
print(f"\n💰 Total Annual CLV : ${clv_full['clv_annual'].sum():,.0f}")
print(f"💰 Avg CLV          : ${clv_full['clv_annual'].mean():,.2f}")
print(f"💰 Top 10% CLV      : ${p90:,.2f}+")

# Save
clv_save = clv_full[[
    'customer_id','source','total_spent','order_count',
    'avg_order','tenure_days','recency_days',
    'clv_annual','clv_3month','clv_segment'
]]

clv_save.to_csv(f'{export_path}\\clv_full_v2.csv', index=False)
clv_save.to_csv(f'{powerbi_path}\\clv_full_v2_powerbi.csv', index=False)

conn = sqlite3.connect(db_path)
clv_save.to_sql('clv_full_v2', conn, if_exists='replace', index=False)
conn.close()

print(f"\n✅ Saved → clv_full_v2.csv + database")

✅ FULL Combined:
   Rows      : 2,034,230
   Customers : 664,229

Source breakdown:
source
bank               6383
data              19960
eco_dataset        1454
ecommerce          5000
kz                22045
olist             96477
online_retail      5878
retail           500000
telco              7032

📊 FULL CLV — 664,229 customers
              count  avg_clv     total_clv
clv_segment                               
Bronze       332113   288.46  9.580236e+07
Gold          99634  2015.53  2.008158e+08
Platinum      66424  8420.96  5.593538e+08
Silver       166058  1350.44  2.242518e+08

💰 Total Annual CLV : $1,080,223,770
💰 Avg CLV          : $1,626.28
💰 Top 10% CLV      : $2,311.51+

✅ Saved → clv_full_v2.csv + database


In [6]:
import sqlite3
import pandas as pd
import os

db_path      = r'C:\Users\user\DSS_Project\database\dss_project.db'
export_path  = r'C:\Users\user\DSS_Project\exports'
powerbi_path = r'C:\Users\user\DSS_Project\exports\powerbi'

conn = sqlite3.connect(db_path)

# 1️⃣ CLV Full V2
clv_save.to_sql('clv_full_v2', conn, if_exists='replace', index=False)
print(f"✅ clv_full_v2          → {len(clv_save):,} rows")

# 2️⃣ CLV Summary by Segment
seg_summary = clv_full.groupby(['clv_segment','source']).agg(
    customers  = ('customer_id', 'count'),
    avg_clv    = ('clv_annual', 'mean'),
    total_clv  = ('clv_annual', 'sum'),
    avg_orders = ('order_count', 'mean'),
    avg_tenure = ('tenure_days', 'mean'),
    avg_recency= ('recency_days', 'mean'),
).round(2).reset_index()

seg_summary.to_sql('clv_segments_summary', conn, 
                    if_exists='replace', index=False)
print(f"✅ clv_segments_summary → {len(seg_summary):,} rows")

# 3️⃣ CLV by Source
source_summary = clv_full.groupby('source').agg(
    customers  = ('customer_id', 'count'),
    avg_clv    = ('clv_annual', 'mean'),
    total_clv  = ('clv_annual', 'sum'),
    avg_orders = ('order_count', 'mean'),
).round(2).reset_index()

source_summary.to_sql('clv_by_source', conn,
                       if_exists='replace', index=False)
print(f"✅ clv_by_source        → {len(source_summary):,} rows")

# 4️⃣ Bootstrap ROI Full
df_boot.to_sql('bootstrap_roi_full', conn,
                if_exists='replace', index=False)
print(f"✅ bootstrap_roi_full   → {len(df_boot):,} rows")

# 5️⃣ Final Project Summary
final_summary = pd.DataFrame({
    'metric': [
        'total_customers',
        'total_annual_clv',
        'avg_clv',
        'median_clv',
        'top10_clv',
        'platinum_customers',
        'gold_customers',
        'silver_customers',
        'bronze_customers',
        'platinum_total_clv',
        'gold_total_clv',
        'silver_total_clv',
        'bronze_total_clv',
        'bootstrap_iterations',
        'overall_roi',
        'best_combo_roi',
        'best_channel',
        'best_segment',
        'best_season',
        'best_region',
        'total_rows',
        'total_sources'
    ],
    'value': [
        f"{len(clv_full):,}",
        f"${clv_full['clv_annual'].sum():,.0f}",
        f"${clv_full['clv_annual'].mean():,.2f}",
        f"${clv_full['clv_annual'].median():,.2f}",
        f"${p90:,.2f}+",
        f"{(clv_full['clv_segment']=='Platinum').sum():,}",
        f"{(clv_full['clv_segment']=='Gold').sum():,}",
        f"{(clv_full['clv_segment']=='Silver').sum():,}",
        f"{(clv_full['clv_segment']=='Bronze').sum():,}",
        f"${clv_full[clv_full['clv_segment']=='Platinum']['clv_annual'].sum():,.0f}",
        f"${clv_full[clv_full['clv_segment']=='Gold']['clv_annual'].sum():,.0f}",
        f"${clv_full[clv_full['clv_segment']=='Silver']['clv_annual'].sum():,.0f}",
        f"${clv_full[clv_full['clv_segment']=='Bronze']['clv_annual'].sum():,.0f}",
        "10,000",
        f"{overall['mean']:+.2f}%",
        f"{best_com['mean']:+.2f}%" if best_com else "N/A",
        f"{best_ch['category']} → {best_ch['mean']:+.2f}%",
        f"{best_seg['category']} → {best_seg['mean']:+.2f}%",
        f"{best_sea['category']} → {best_sea['mean']:+.2f}%",
        f"{best_reg['category']} → {best_reg['mean']:+.2f}%",
        f"{len(combined_full):,}",
        f"{combined_full['source'].nunique()}"
    ]
})

final_summary.to_sql('project_final_summary', conn,
                      if_exists='replace', index=False)
print(f"✅ project_final_summary→ {len(final_summary):,} rows")

# Save CSVs
clv_save.to_csv(f'{export_path}\\clv_full_v2.csv', index=False)
clv_save.to_csv(f'{powerbi_path}\\clv_full_v2_powerbi.csv', index=False)
seg_summary.to_csv(f'{export_path}\\clv_segments_summary.csv', index=False)
seg_summary.to_csv(f'{powerbi_path}\\clv_segments_summary_powerbi.csv', index=False)
source_summary.to_csv(f'{export_path}\\clv_by_source.csv', index=False)
source_summary.to_csv(f'{powerbi_path}\\clv_by_source_powerbi.csv', index=False)
final_summary.to_csv(f'{export_path}\\project_final_summary.csv', index=False)
final_summary.to_csv(f'{powerbi_path}\\project_final_summary_powerbi.csv', index=False)

conn.close()

# Verify
print(f"\n{'='*60}")
print(f"✅ DATABASE UPDATED SUCCESSFULLY")
print(f"{'='*60}")

conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print(f"\n📊 Total Tables: {len(tables)}")
for t in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {t[0]}")
    count = cursor.fetchone()[0]
    print(f"   ✅ {t[0]:40} → {count:,} rows")
conn.close()

print(f"\n{'='*60}")
print(f"📁 FILES SAVED:")
print(f"{'='*60}")
print(f"   exports/clv_full_v2.csv")
print(f"   exports/clv_segments_summary.csv")
print(f"   exports/clv_by_source.csv")
print(f"   exports/project_final_summary.csv")
print(f"   powerbi/clv_full_v2_powerbi.csv")
print(f"   powerbi/clv_segments_summary_powerbi.csv")
print(f"   powerbi/clv_by_source_powerbi.csv")
print(f"   powerbi/project_final_summary_powerbi.csv")

✅ clv_full_v2          → 664,229 rows
✅ clv_segments_summary → 33 rows
✅ clv_by_source        → 9 rows
✅ bootstrap_roi_full   → 36 rows
✅ project_final_summary→ 22 rows

✅ DATABASE UPDATED SUCCESSFULLY

📊 Total Tables: 30
   ✅ retail                                   → 1,000,000 rows
   ✅ ecommerce                                → 22,049 rows
   ✅ customer_features                        → 5,000 rows
   ✅ marketing_features                       → 60,000 rows
   ✅ customer_segments                        → 5,000 rows
   ✅ association_rules                        → 2,900 rows
   ✅ churn_analysis                           → 1,000,000 rows
   ✅ telco_churn                              → 7,043 rows
   ✅ bank_churn                               → 10,000 rows
   ✅ sales_forecast                           → 6 rows
   ✅ roi_by_channel                           → 5 rows
   ✅ roi_by_region                            → 7 rows
   ✅ recommendations                          → 12 rows
   ✅ product_la